In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.append(str(project_root))

In [2]:
from src.model import load_model

feature_extractor, model = load_model()

print("Model loaded successfully!")

e:\Audio-Environmental-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Audio-Environmental-Classification\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--MIT--ast-finetuned-audioset-10-10-0.4593. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adminis

Model loaded successfully!


In [3]:
number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Parameters:",
    f"{number_of_parameters:,}"
)

Parameters: 86,594,063


In [4]:
print(
    "Model size:",
    f"{number_of_parameters * 4 / (1024 ** 2):.2f} MB"
)

Model size: 330.33 MB


In [5]:
import librosa

audio_path = project_root / "data" / "raw" / "4-189838-A-22.wav"

audio, sample_rate = librosa.load(
    audio_path,
    sr=16000,
    mono=True
)

print("Sampling rate:", sample_rate)
print("Samples:", len(audio))
print("Duration:", len(audio) / sample_rate)

Sampling rate: 16000
Samples: 80000
Duration: 5.0


In [6]:
inputs = feature_extractor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

print(inputs.keys())

KeysView({'input_values': tensor([[[-0.8699, -1.0158, -0.6389,  ...,  0.3436,  0.2356, -0.1261],
         [-0.9255, -1.0423, -0.6655,  ...,  0.2519,  0.1801, -0.1670],
         [-0.7133, -1.0767, -0.6999,  ...,  0.1261,  0.0693, -0.2888],
         ...,
         [ 0.4670,  0.4670,  0.4670,  ...,  0.4670,  0.4670,  0.4670],
         [ 0.4670,  0.4670,  0.4670,  ...,  0.4670,  0.4670,  0.4670],
         [ 0.4670,  0.4670,  0.4670,  ...,  0.4670,  0.4670,  0.4670]]])})


In [8]:
import torch

print("PyTorch:", torch.__version__)

PyTorch: 2.14.1+cpu


In [9]:
with torch.no_grad():
    outputs = model(**inputs)

print("Inference completed!")
print("Logits shape:", outputs.logits.shape)

Inference completed!
Logits shape: torch.Size([1, 527])


In [10]:
predicted_id = outputs.logits.argmax(dim=-1).item()

print("Predicted ID:", predicted_id)

Predicted ID: 67


In [11]:
predicted_label = model.config.id2label[predicted_id]

print("Predicted sound:", predicted_label)

Predicted sound: Applause


In [12]:
probabilities = torch.softmax(
    outputs.logits,
    dim=-1
)

top_probabilities, top_indices = torch.topk(
    probabilities,
    k=5,
    dim=-1
)

In [13]:
for probability, index in zip(
    top_probabilities[0],
    top_indices[0]
):
    label = model.config.id2label[index.item()]

    print(
        f"{label}: {probability.item() * 100:.2f}%"
    )

Applause: 89.81%
Speech: 3.34%
Animal: 0.52%
Clapping: 0.51%
Music: 0.50%


In [14]:
def predict_audio(audio_path):
    # Load audio
    audio, sample_rate = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    # Prepare input
    inputs = feature_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    # Run model
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    probabilities = torch.softmax(outputs.logits, dim=-1)

    # Get top 5
    top_probabilities, top_indices = torch.topk(
        probabilities,
        k=5,
        dim=-1
    )

    print(f"File: {audio_path.name}")
    print("\nTop 5 predictions:")

    for probability, index in zip(
        top_probabilities[0],
        top_indices[0]
    ):
        label = model.config.id2label[index.item()]
        percentage = probability.item() * 100

        print(f"{label}: {percentage:.2f}%")

In [15]:
from pathlib import Path

audio_files = list(Path("../data/raw").glob("*.wav"))

print("Number of WAV files:", len(audio_files))

Number of WAV files: 250


In [16]:
predict_audio(audio_files[1])

File: 1-101296-B-19.wav

Top 5 predictions:
Thunder: 54.91%
Thunderstorm: 29.02%
Rain: 7.96%
Rain on surface: 1.22%
Raindrop: 1.08%


In [17]:
predict_audio(audio_files[2])

File: 1-104089-A-22.wav

Top 5 predictions:
Clapping: 79.10%
Applause: 15.51%
Sound effect: 1.78%
Tap: 0.51%
Ping: 0.18%


In [18]:
predict_audio(audio_files[3])

File: 1-110537-A-22.wav

Top 5 predictions:
Applause: 90.14%
Clapping: 2.17%
Rain on surface: 1.08%
Crowd: 1.02%
Rain: 0.67%


In [19]:
for audio_path in audio_files[:5]:
    predict_audio(audio_path)
    print("-" * 50)

File: 1-100032-A-0.wav

Top 5 predictions:
Bark: 18.76%
Animal: 17.97%
Dog: 9.85%
Bow-wow: 8.36%
Domestic animals, pets: 7.12%
--------------------------------------------------
File: 1-101296-B-19.wav

Top 5 predictions:
Thunder: 54.91%
Thunderstorm: 29.02%
Rain: 7.96%
Rain on surface: 1.22%
Raindrop: 1.08%
--------------------------------------------------
File: 1-104089-A-22.wav

Top 5 predictions:
Clapping: 79.10%
Applause: 15.51%
Sound effect: 1.78%
Tap: 0.51%
Ping: 0.18%
--------------------------------------------------
File: 1-110537-A-22.wav

Top 5 predictions:
Applause: 90.14%
Clapping: 2.17%
Rain on surface: 1.08%
Crowd: 1.02%
Rain: 0.67%
--------------------------------------------------
File: 1-115521-A-19.wav

Top 5 predictions:
Thunder: 48.75%
Thunderstorm: 36.57%
Rain: 5.83%
Rain on surface: 2.10%
Explosion: 1.25%
--------------------------------------------------
